In [1]:
import cv2
import supervision as sv
from rfdetr import RFDETRSegMedium
from rfdetr.assets.coco_classes import COCO_CLASSES
from trackers import ByteTrackTracker

model = RFDETRSegMedium()  # local weights; auto-picks CUDA/MPS/CPU when available
# optional if you hit MPS issues:
# model.optimize_for_inference(compile=False, inplace=True)

tracker = ByteTrackTracker()
mask_annotator = sv.MaskAnnotator(opacity=0.5)
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()
trace_annotator = sv.TraceAnnotator(trace_length=30, thickness=2)

cap = cv2.VideoCapture("../data/zeebra.mp4")
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0

out = cv2.VideoWriter(
    "output-seg.mp4",
    cv2.VideoWriter_fourcc(*"avc1"),
    fps,
    (width, height),
)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # returns sv.Detections with boxes + masks (no inference package)
    detections = model.predict(frame, threshold=0.5)
    detections = tracker.update(detections)

    labels = [
        COCO_CLASSES[cid] if cid is not None else ""
        for cid in detections.class_id
    ]

    frame = mask_annotator.annotate(scene=frame, detections=detections)
    frame = box_annotator.annotate(scene=frame, detections=detections)
    frame = label_annotator.annotate(scene=frame, detections=detections, labels=labels)
    frame = trace_annotator.annotate(scene=frame, detections=detections)

    out.write(frame)

cap.release()
out.release()

/Users/krish/Workspace/mot-exp/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[2026-10-19 11:13:47] [INFO] rf-detr - Downloading pretrained weights for /Users/krish/.roboflow/models/rf-detr-seg-medium.pt


/Users/krish/Workspace/mot-exp/.venv/lib/python3.13/site-packages/trackers/core/sort/tracker.py:26: FutureWarning: target=None is deprecated since `v0.8`; use `TargetMode.NOTIFY` instead. Will be removed in `v1.0`.
  @deprecated(target=None, deprecated_in="0.4", remove_in="1.0")


/Users/krish/.roboflow/models/rf-detr-seg-medium.pt:   0%|          | 0.00/136M [00:00<?, ?iB/s]

[2026-10-19 11:14:13] [INFO] rf-detr - MD5 validation successful for /Users/krish/.roboflow/models/rf-detr-seg-medium.pt


[2026-10-19 11:14:13] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-10-19 11:14:13] [WARNING] rf-detr - Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-10-19 11:14:14] [INFO] rf-detr - File /Users/krish/.roboflow/models/rf-detr-seg-medium.pt already exists with correct MD5 hash.


[2026-10-19 11:14:14] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. For full GPU throughput (e.g. ~8x on T4 via FP16 Tensor Cores), call model.inference(dtype=torch.float16).
